# Reproduction of Persian miniature dataset creation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshaSrirangam/stable-diffusion-rebuilt/blob/main/notebooks/build_dataset.ipynb)

## Environment/setup

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
# clone repo
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

fatal: destination path 'stable-diffusion-rebuilt' already exists and is not an empty directory.
/content/stable-diffusion-rebuilt
Already up to date.


In [3]:
# install dependencies (skip torch/numpy to avoid overwriting Colab's preinstalled packages)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stable-diffusion-rebuilt (pyproject.toml) ... done


## Create/download dataset

In [4]:
# either build from scratch or download from HF
BUILD_FROM_SCRATCH = False # if True, a GPU runtime is preffered for BLIP captioning

if BUILD_FROM_SCRATCH:
    !python scripts/scrape.py
    !python scripts/filter_persian.py --data-dir data/persian/raw
    !python scripts/caption.py --data-dir data/persian/raw
else:
    from huggingface_hub import snapshot_download
    snapshot_download("HarshaSrirangam/persian-miniatures", repo_type="dataset",
                      local_dir="data/persian/raw")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5243 files:   0%|          | 0/5243 [00:00<?, ?it/s]

In [ ]:
!ls data/persian/raw # should contain images/ and metadata.jsonl (and metadata_raw.jsonl, the uncleaned BLIP captions)
!ls data/persian/raw/images | wc -l # ~5243

images	metadata.jsonl	metadata_raw.jsonl  README.md
5239


In [6]:
# create train/eval splits
import json
import random
from pathlib import Path
import shutil

raw_dir = Path("data/persian/raw")
train_size = 1300
eval_size = 165
seed = 42

# sample TRAIN_SIZE + EVAL_SIZE images from raw/, then split
paths = sorted((raw_dir / "images").glob("*.jpg"))
sample = random.Random(seed).sample(paths, train_size + eval_size)
splits = {"train": sample[:train_size], "eval": sample[train_size:]}

# get captions by file_name
captions = {}
with open(raw_dir / "metadata.jsonl") as meta:
    for line in meta:
        row = json.loads(line)
        captions[row["file_name"]] = row["text"]

for split, split_files in splits.items():
    out_dir = raw_dir.parent / split # data/persian/<split>/
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    with open(out_dir / "metadata.jsonl", "w") as meta:
        for p in split_files:
            shutil.copy(p, out_dir / "images" / p.name)
            fn = f"images/{p.name}"
            meta.write(json.dumps({"file_name": fn, "text": captions[fn]}) + "\n")

In [8]:
!ls data/persian/train/images | wc -l # train_size
!ls data/persian/eval/images | wc -l # eval_size

1300
165
